In [ ]:
import ctypes
import numpy as np
import time
import cv2
import gc
import os
import zlib
from sdlarch_rl import make
import pygame
from IPython.display import Audio
from sdlarch_rl.utils.discretizer import MainDiscretizer

env = make("GranTurismo3-Ps2")

obs, info = env.reset()

height, width, _ = obs.shape
p_outside_left = {
    "x1": int(width * 0.25),
    "x2": int(width * 0.35),
    "y1": int(height * 0.5),
    "y2": int(height * 0.65),
}

p_outside_right = {
    "x1": int(width * 0.65),
    "x2": int(width * 0.75),
    "y1": int(height * 0.5),
    "y2": int(height * 0.65),
}

def is_touching_wall(obs, p_outside, white_threshold=0.15):

        height, width, _ = obs.shape

        roi = obs[p_outside['y1']:p_outside['y2'], p_outside['x1']:p_outside['x2']]
    
        lab = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        roi_clahe = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    
        hsv = cv2.cvtColor(roi_clahe, cv2.COLOR_BGR2HSV)
    
        lower_white = np.array([0, 0, 50])
        upper_white = np.array([0, 0, 70])
        white_mask = cv2.inRange(hsv, lower_white, upper_white)
    
        cv2.imshow("Wall Mask", white_mask)
    
        white_ratio = np.sum(white_mask > 0) / (roi.shape[0] * roi.shape[1])
    

        return white_ratio > white_threshold

pygame.init()


SCREEN_WIDTH = 1920
SCREEN_HEIGHT = int(SCREEN_WIDTH * (height / width))

p_outside_left_screen = {
    "x1": int(SCREEN_WIDTH * 0.25),
    "x2": int(SCREEN_WIDTH * 0.35),
    "y1": int(SCREEN_HEIGHT * 0.5),
    "y2": int(SCREEN_HEIGHT * 0.65),
}

p_outside_right_screen = {
    "x1": int(SCREEN_WIDTH * 0.65),
    "x2": int(SCREEN_WIDTH * 0.75),
    "y1": int(SCREEN_HEIGHT * 0.5),
    "y2": int(SCREEN_HEIGHT * 0.65),
}
window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()


count = 0
global initial_state
initial_state = None

framerate = env.unwrapped.em.get_frame_rate()

frame_time = 1.0 / framerate
last_time = time.time()

INVERT_AXIS = False

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True
            
    keys = pygame.key.get_pressed()

    env.render()

    action = np.zeros(16, dtype=np.uint8)

    # pad
    if keys[pygame.K_UP]:
        if INVERT_AXIS:
            action[7] = 1
        else:
            action[4] = 1
    if keys[pygame.K_DOWN]:
        if INVERT_AXIS:
            action[6] = 1
        else:
            action[5] = 1
    if keys[pygame.K_LEFT]:
        if INVERT_AXIS:
            action[4] = 1
        else:
            action[6] = 1
    if keys[pygame.K_RIGHT]:
        if INVERT_AXIS:
            action[5] = 1
        else:
            action[7] = 1

    # buttons
    if keys[pygame.K_x]:
        action[0] = 1
    if keys[pygame.K_c]:
        action[1] = 1
    if keys[pygame.K_y]:
        action[8] = 1
    if keys[pygame.K_z]:
        action[9] = 1
    if keys[pygame.K_q]:
        action[10] = 1
    if keys[pygame.K_w]:
        action[11] = 1
    if keys[pygame.K_RETURN]:
        action[3] = 1

    if keys[pygame.K_BACKSPACE]:
        print("== DONE ==")
        done = True

    img, rew, done, _, info = env.step(action)

    # frame rate
    now = time.time()
    sleep_time = frame_time - (now - last_time)
    if sleep_time > 0:
        time.sleep(sleep_time)
    last_time = now

    if rew < 0:
        print(rew);

    # off_track_left = is_touching_wall(img, p_outside_left)
    # off_track_left = False
    # off_track_right = is_touching_wall(img, p_outside_right)

    # if off_track_left or off_track_right:
    #     print("Off Track")

    # off_track = off_track_right or off_track_left
    
    # img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))

    # color = (255, 0, 0) if off_track else (0,255,0)  # outside road

    # # color=(0,255,0)
    # thickness=3
    # cv2.rectangle(img, (p_outside_left_screen['x1'], p_outside_left_screen['y1']), (p_outside_left_screen['x2'],p_outside_left_screen['y2']), color, thickness)
    # cv2.rectangle(img, (p_outside_right_screen['x1'], p_outside_right_screen['y1']), (p_outside_right_screen['x2'],p_outside_right_screen['y2']), color, thickness)
    surface = pygame.surfarray.make_surface(np.transpose(img, (1, 0, 2)))

    
    # rect = pygame.Rect(p_outside_screen['x1'], p_outside_screen['y1'], p_outside_screen['x2'], p_outside_screen['y2'])
    # pygame.draw.rect(surface, color, rect, thickness)

    window.blit(surface, (0, 0))
    pygame.display.update()

    count += 1

    if done:
        break

    

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.2
-0.9988974358974358
-0.9988717948717949
-0.9988717948717949
-0.9988717948717949
-0.9988717948717949
-0.9988974358974358
-0.9988974358974358
-0.9988974358974358
-0.9988974358974358
-0.9988974358974358
-0.9988974358974358
-0.9988974358974358
-0.9988974358974358
-0.9988974358974358
-0.9988974358974358
-0.9989230769230769
-0.9989230769230769
-0.9989230769230769
-0.9989230769230769
-0.9989230769230769
-0.9989230769230769
-0.9989230769230769
-0